# AURA stylist LLM — LoRA fine-tune of Qwen2.5-1.5B-Instruct

The AURA backend builds complete outfits only from garments the user owns and ranks them with the Polyvore-trained compatibility model plus occasion, weather, preference and feedback signals. This notebook trains the **language model that makes the final call**: given the user's occasion, weather, self-described body shape, height, skin tone and saved preferences, it picks the best of up to five numbered options and explains why. It never sees or writes database IDs, so it cannot invent clothes.

**Where the "fashion sense" in the training data comes from.** Every training example is built from a *real* Polyvore outfit that a person composed (the correct answer) and four distractors:
- the same outfit with one garment swapped for another garment of the same type (hard),
- the same outfit with two garments swapped,
- a fully random outfit,
- and — whenever the outfit title states an occasion or season — a *real, well-composed outfit for a different occasion or season*, so the model must use the context, not only visual harmony.

Occasions and seasons come from the human-written outfit titles (e.g. "office chic", "winter date night"). Avoided colours from the user's preferences are always absent from the correct answer. Body shape, height and skin tone are included so the model learns to read them, but Polyvore has no body-type labels, so **no claim is made that the model learned body-type styling** — it uses its pretrained knowledge there.

**Safe to re-run.** Every stage saves to `MyDrive/AURA/stylist_llm/` as it finishes (examples, each evaluation, a training checkpoint every 100 steps, the adapter, the GGUF). If Colab disconnects or runs out of GPU time, open the notebook again and choose Run all: finished stages are skipped and training resumes from the newest checkpoint.

**Before you run:** finish `AURA_Outfit_Compatibility_Polyvore_Colab.ipynb` first (this notebook reuses its fingerprints and model). Runtime → **T4 GPU** → Run all. Expected time on a free T4: data ≈10 min, base-model evaluation ≈10 min, LoRA training ≈40–70 min, evaluation ≈10 min, GGUF export ≈15 min.

**Outputs** in `MyDrive/AURA/stylist_llm/`: `aura-stylist-q4_k_m.gguf` (copy to `Capstone_Agents/model_artifacts/llm/`), the LoRA adapter, `evaluation.json`, and the train/valid/test JSONL files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, random, re, math
# Run Keras (the compatibility model) on the PyTorch backend: TensorFlow and PyTorch in one
# process can clash, and the shared model code is backend-neutral (identical scores).
os.environ['KERAS_BACKEND'] = 'torch'
os.environ['USE_TF'] = '0'   # keep Hugging Face Transformers on PyTorch only
from pathlib import Path
import numpy as np

AURA = Path('/content/drive/MyDrive/AURA')
COMPAT_DIR = AURA / 'outfit_compat'
CACHE = AURA / 'polyvore_cache'
OUT = AURA / 'stylist_llm'; OUT.mkdir(parents=True, exist_ok=True)
DATA = Path('/content/polyvore'); DATA.mkdir(exist_ok=True)
POLYVORE_REPO = 'Stylique/Polyvore'
SPLIT = 'nondisjoint'
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
SEED = 7
N_TRAIN, N_VALID, N_TEST = 12000, 400, 600
random.seed(SEED); np.random.seed(SEED)
assert (COMPAT_DIR / 'outfit_compat.weights.h5').exists(), 'Run the outfit-compatibility notebook first.'
assert list(CACHE.glob('embeddings_*.npz')), 'Garment fingerprints missing; run the compatibility notebook first.'

%pip -q install "transformers>=4.45,<5" "peft>=0.13,<1" "accelerate>=0.34" "datasets>=2.20" sentencepiece
os.makedirs('/content/aura_ml', exist_ok=True)
open('/content/aura_ml/__init__.py', 'w').close()
sys.path.insert(0, '/content')


## Shared code
These cells write `ml/outfit_compat.py` and `ml/stylist_prompt.py` from the backend byte-for-byte (a backend test checks this), so the prompt the model is trained on is exactly the prompt the app sends.


In [ ]:
%%writefile aura_ml/outfit_compat.py
"""AURA outfit-compatibility transformer (trained from scratch on Polyvore Outfits).

Each garment is represented by the 512-d ``fashion_feature_normalization``
embedding from AURA's own EfficientNetV2 garment CNN plus its broad AURA
category. A small transformer encoder reads the whole outfit as a set and
predicts whether the pieces belong together.

This file is shared verbatim by the backend and
``training/AURA_Outfit_Compatibility_Polyvore_Colab.ipynb``. Keep it free of
backend imports so it can run unchanged in Colab.
"""
from __future__ import annotations

from typing import Any, Sequence

import numpy as np

EMBEDDING_DIM = 512
MAX_ITEMS = 16
EMBEDDING_LAYER = "fashion_feature_normalization"

# 0 is padding, the CLS token uses its own learned vector.
AURA_CATEGORIES = ("Top", "Bottom", "Dress", "Footwear", "Outerwear", "Accessory")
CATEGORY_TO_ID = {name: index + 1 for index, name in enumerate(AURA_CATEGORIES)}

# Polyvore Outfits "semantic_category" -> AURA wardrobe category.
POLYVORE_TO_AURA = {
    "tops": "Top",
    "bottoms": "Bottom",
    "all-body": "Dress",
    "shoes": "Footwear",
    "outerwear": "Outerwear",
    "bags": "Accessory",
    "accessories": "Accessory",
    "jewellery": "Accessory",
    "hats": "Accessory",
    "scarves": "Accessory",
    "sunglasses": "Accessory",
}

DEFAULT_CONFIG: dict[str, Any] = {
    "embedding_dim": EMBEDDING_DIM,
    "max_items": MAX_ITEMS,
    "model_dim": 256,
    "num_heads": 4,
    "num_layers": 4,
    "ff_dim": 512,
    "dropout": 0.1,
    "input_noise": 0.05,
    "feature_dropout": 0.1,
}


def category_id(category: str | None) -> int:
    """Map an AURA category name to its integer id (unknown -> Accessory)."""
    return CATEGORY_TO_ID.get(str(category or ""), CATEGORY_TO_ID["Accessory"])


def encode_outfits(
    outfits: Sequence[Sequence[tuple[np.ndarray, str]]],
    max_items: int = MAX_ITEMS,
) -> tuple[np.ndarray, np.ndarray]:
    """Pack outfits of (embedding, AURA category) pairs into padded arrays."""
    features = np.zeros((len(outfits), max_items, EMBEDDING_DIM), dtype=np.float32)
    categories = np.zeros((len(outfits), max_items), dtype=np.int32)
    for row, outfit in enumerate(outfits):
        for column, (embedding, category) in enumerate(list(outfit)[:max_items]):
            features[row, column] = np.asarray(embedding, dtype=np.float32)
            categories[row, column] = category_id(category)
    return features, categories


def build_model(config: dict[str, Any] | None = None):
    """Build the Keras model. Weights are random until trained or loaded."""
    import keras
    from keras import layers, ops

    cfg = {**DEFAULT_CONFIG, **(config or {})}
    d_model = int(cfg["model_dim"])

    @keras.saving.register_keras_serializable(package="AURAOutfitCompat")
    class OutfitEncoder(layers.Layer):
        def __init__(self, cfg: dict[str, Any], **kwargs: Any) -> None:
            super().__init__(**kwargs)
            self.cfg = dict(cfg)
            self.project = layers.Dense(d_model, name="item_projection")
            self.category_embedding = layers.Embedding(
                len(AURA_CATEGORIES) + 1, d_model, name="category_embedding"
            )
            self.noise = layers.GaussianNoise(float(cfg["input_noise"]))
            self.feature_dropout = layers.Dropout(float(cfg["feature_dropout"]))
            self.blocks = []
            for index in range(int(cfg["num_layers"])):
                self.blocks.append({
                    "norm_1": layers.LayerNormalization(epsilon=1e-6, name=f"block{index}_norm1"),
                    "attention": layers.MultiHeadAttention(
                        num_heads=int(cfg["num_heads"]),
                        key_dim=d_model // int(cfg["num_heads"]),
                        dropout=float(cfg["dropout"]),
                        name=f"block{index}_attention",
                    ),
                    "drop_1": layers.Dropout(float(cfg["dropout"])),
                    "norm_2": layers.LayerNormalization(epsilon=1e-6, name=f"block{index}_norm2"),
                    "ff_1": layers.Dense(int(cfg["ff_dim"]), activation="gelu", name=f"block{index}_ff1"),
                    "ff_2": layers.Dense(d_model, name=f"block{index}_ff2"),
                    "drop_2": layers.Dropout(float(cfg["dropout"])),
                })
            self.final_norm = layers.LayerNormalization(epsilon=1e-6, name="final_norm")

        def build(self, input_shape: Any) -> None:
            self.cls_token = self.add_weight(
                name="cls_token", shape=(1, 1, d_model), initializer="random_normal", trainable=True
            )
            super().build(input_shape)

        def call(self, features: Any, categories: Any, training: bool | None = None) -> Any:
            features = self.noise(features, training=training)
            features = self.feature_dropout(features, training=training)
            tokens = self.project(features) + self.category_embedding(categories)
            batch = ops.shape(tokens)[0]
            cls = ops.broadcast_to(self.cls_token, (batch, 1, d_model))
            tokens = ops.concatenate([cls, tokens], axis=1)
            valid = ops.concatenate(
                [ops.ones((batch, 1), dtype="bool"), ops.greater(categories, 0)], axis=1
            )
            mask = ops.logical_and(ops.expand_dims(valid, 1), ops.expand_dims(valid, 2))
            x = tokens
            for block in self.blocks:
                y = block["norm_1"](x)
                y = block["attention"](y, y, attention_mask=mask, training=training)
                x = x + block["drop_1"](y, training=training)
                y = block["norm_2"](x)
                y = block["ff_2"](block["ff_1"](y))
                x = x + block["drop_2"](y, training=training)
            return self.final_norm(x)[:, 0, :]

        def get_config(self) -> dict[str, Any]:
            return {**super().get_config(), "cfg": self.cfg}

    features = keras.Input((int(cfg["max_items"]), int(cfg["embedding_dim"])), name="features")
    categories = keras.Input((int(cfg["max_items"]),), dtype="int32", name="categories")
    pooled = OutfitEncoder(cfg, name="outfit_encoder")(features, categories)
    hidden = layers.Dense(d_model, activation="gelu", name="head_hidden")(pooled)
    hidden = layers.Dropout(float(cfg["dropout"]))(hidden)
    logit = layers.Dense(1, name="compatibility_logit")(hidden)
    return keras.Model({"features": features, "categories": categories}, logit, name="aura_outfit_compat")


def predict_scores(model: Any, features: np.ndarray, categories: np.ndarray, batch_size: int = 512) -> np.ndarray:
    """Return compatibility probabilities in [0, 1] for packed outfits."""
    logits = model.predict(
        {"features": features, "categories": categories}, batch_size=batch_size, verbose=0
    )
    logits = np.asarray(logits, dtype=np.float64).reshape(-1)
    return 1.0 / (1.0 + np.exp(-logits))


In [ ]:
%%writefile aura_ml/stylist_prompt.py
"""Prompt and answer format for the AURA stylist language model.

The stylist LLM never sees or writes database item IDs. The server sends up to
five complete, owned outfit options (already checked for ownership and
completeness); the model answers with the option number it recommends and a
short reason. This keeps every suggestion inside the user's real wardrobe.

This file is shared verbatim by the backend and
``training/AURA_Stylist_LLM_LoRA_Colab.ipynb`` so training prompts and
inference prompts are byte-identical.
"""
from __future__ import annotations

import json
import re
from typing import Any, Sequence

MAX_OPTIONS = 5
REASON_MAX_CHARS = 420

SYSTEM_PROMPT = (
    "You are AURA, an expert personal fashion stylist. You are given the user's context "
    "(occasion, weather, self-described body shape, height, skin tone and saved preferences) "
    "and numbered outfit options built only from clothes the user owns. Choose the single "
    "option that best suits the occasion and weather, has the most coherent colours and "
    "proportions, and respects the user's preferences. visual_match is a learned estimate of "
    "how well the pieces look together. Use body shape only as optional proportion guidance, "
    "never as a criticism. Mention only garments and attributes listed in the chosen option. "
    'Reply with JSON only: {"choice": <option number>, "reason": "<one or two sentences>"}.'
)

RESPONSE_SCHEMA: dict[str, Any] = {
    "type": "object",
    "properties": {
        "choice": {"type": "integer", "minimum": 1, "maximum": MAX_OPTIONS},
        "reason": {"type": "string", "maxLength": REASON_MAX_CHARS},
    },
    "required": ["choice", "reason"],
}

_ITEM_FIELDS = ("category", "type", "color", "pattern", "fabric", "fit")
_PREFERENCE_FIELDS = (
    "preferred_colors", "avoided_colors", "preferred_styles", "disliked_styles", "fit_preference",
)


def visual_match_label(probability: float | None) -> str:
    if probability is None:
        return "unknown"
    if probability >= 0.7:
        return "high"
    if probability >= 0.4:
        return "medium"
    return "low"


def _clean(value: Any) -> Any:
    if isinstance(value, str):
        value = value.strip()
        return value or None
    if isinstance(value, (list, tuple)):
        cleaned = [_clean(entry) for entry in value]
        return [entry for entry in cleaned if entry not in (None, [], {})] or None
    return value


def build_user_payload(context: dict[str, Any], options: Sequence[dict[str, Any]]) -> str:
    """Serialize context and options deterministically (compact JSON)."""
    weather = context.get("weather") or {}
    preferences = context.get("preferences") or {}
    payload: dict[str, Any] = {
        "occasion": _clean(context.get("occasion")) or "casual",
        "weather": {
            "temperature_c": weather.get("temperature_c"),
            "condition": _clean(weather.get("condition")) or "unknown",
        },
    }
    for key in ("body_shape", "height_cm", "skin_tone"):
        value = _clean(context.get(key))
        if value is not None:
            payload[key] = value
    saved = {key: _clean(preferences.get(key)) for key in _PREFERENCE_FIELDS}
    saved = {key: value for key, value in saved.items() if value is not None}
    if saved:
        payload["preferences"] = saved
    payload["options"] = []
    for number, option in enumerate(list(options)[:MAX_OPTIONS], start=1):
        items = []
        for item in option.get("items", []):
            entry = {key: _clean(item.get(key)) for key in _ITEM_FIELDS}
            items.append({key: value for key, value in entry.items() if value is not None})
        payload["options"].append({
            "option": number,
            "visual_match": visual_match_label(option.get("visual_match")),
            "items": items,
        })
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))


def build_messages(context: dict[str, Any], options: Sequence[dict[str, Any]]) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_payload(context, options)},
    ]


def format_answer(choice: int, reason: str) -> str:
    return json.dumps({"choice": int(choice), "reason": reason.strip()[:REASON_MAX_CHARS]}, ensure_ascii=False)


def parse_answer(raw: str, option_count: int) -> tuple[int, str]:
    """Parse and validate the model's JSON answer; raise ValueError if unusable."""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", str(raw).strip(), flags=re.IGNORECASE)
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end <= start:
        raise ValueError("No JSON object in stylist answer")
    decoded = json.loads(text[start:end + 1])
    if not isinstance(decoded, dict):
        raise ValueError("Stylist answer is not an object")
    choice = decoded.get("choice")
    if isinstance(choice, str) and choice.strip().isdigit():
        choice = int(choice.strip())
    if not isinstance(choice, int) or isinstance(choice, bool) or not 1 <= choice <= option_count:
        raise ValueError("Stylist choice is outside the offered options")
    reason = decoded.get("reason")
    if not isinstance(reason, str) or len(reason.strip()) < 10:
        raise ValueError("Stylist reason is missing")
    return choice, reason.strip()[:REASON_MAX_CHARS]


## 1. Load Polyvore metadata, fingerprints and the compatibility model

In [ ]:
import keras
assert keras.backend.backend() == 'torch', 'Restart the runtime and run from the top (KERAS_BACKEND must be set first).'
from huggingface_hub import snapshot_download
from aura_ml.outfit_compat import build_model, predict_scores, encode_outfits, POLYVORE_TO_AURA, MAX_ITEMS
from aura_ml import stylist_prompt as sp

snapshot_download(POLYVORE_REPO, repo_type='dataset', local_dir=str(DATA),
                  allow_patterns=[f'{SPLIT}/train.json', f'{SPLIT}/valid.json', f'{SPLIT}/test.json',
                                  'polyvore_item_metadata.json', 'polyvore_outfit_titles.json', 'categories.csv'])
item_meta = json.load(open(DATA / 'polyvore_item_metadata.json'))
outfit_titles = json.load(open(DATA / 'polyvore_outfit_titles.json'))
category_rows = {}
for line in open(DATA / 'categories.csv', encoding='utf-8'):
    parts = [part.strip() for part in line.strip().split(',')]
    if len(parts) >= 3 and parts[0].isdigit():
        category_rows[parts[0]] = (parts[1], parts[2])

ids, vectors = [], []
for shard in sorted(CACHE.glob('embeddings_*.npz')):
    data = np.load(shard); ids.extend(data['ids'].tolist()); vectors.append(data['vectors'])
EMB = np.concatenate(vectors).astype(np.float32)
INDEX = {item: i for i, item in enumerate(ids)}
config = json.load(open(COMPAT_DIR / 'outfit_compat.config.json'))
compat_model = build_model(config['model_config'])
compat_model.load_weights(str(COMPAT_DIR / 'outfit_compat.weights.h5'))
print('fingerprints', EMB.shape, '| compat test metrics', config['test_metrics'])


## 2. Turn Polyvore looks into AURA-style outfits with occasion and season context

In [ ]:
COLORS = ['black', 'white', 'grey', 'gray', 'navy', 'blue', 'red', 'pink', 'green', 'olive', 'khaki',
          'yellow', 'mustard', 'orange', 'coral', 'purple', 'lilac', 'burgundy', 'maroon', 'brown', 'tan',
          'beige', 'camel', 'cream', 'ivory', 'nude', 'gold', 'silver', 'teal', 'turquoise', 'mint']
PATTERNS = {'striped': 'Striped', 'stripe': 'Striped', 'floral': 'Floral', 'plaid': 'Plaid', 'check': 'Checked',
            'polka': 'Polka dot', 'leopard': 'Animal print', 'print': 'Printed', 'printed': 'Printed', 'graphic': 'Graphic'}
FABRICS = {'denim': 'Denim', 'leather': 'Leather', 'wool': 'Wool', 'cashmere': 'Cashmere', 'silk': 'Silk',
           'linen': 'Linen', 'cotton': 'Cotton', 'suede': 'Suede', 'knit': 'Knit', 'velvet': 'Velvet', 'satin': 'Satin', 'lace': 'Lace'}
FITS = {'oversized': 'Oversized', 'slim': 'Slim', 'skinny': 'Skinny', 'relaxed': 'Relaxed', 'wide-leg': 'Wide',
        'wide leg': 'Wide', 'fitted': 'Fitted', 'cropped': 'Cropped', 'boyfriend': 'Relaxed', 'bodycon': 'Fitted'}

def first_match(text, table):
    for key, value in table.items():
        if re.search(rf'\b{re.escape(key)}\b', text):
            return value
    return None

def garment(item_id):
    meta = item_meta.get(item_id, {})
    text = f"{meta.get('title', '')} {meta.get('url_name', '')}".lower()
    name, semantic = category_rows.get(str(meta.get('category_id')), (None, meta.get('semantic_category')))
    semantic = meta.get('semantic_category') or semantic
    color = next((c for c in COLORS if re.search(rf'\b{c}\b', text)), None)
    return {
        'item_id': item_id,
        'category': POLYVORE_TO_AURA.get(semantic, 'Accessory'),
        'type': (name or semantic or 'item').strip().title(),
        'color': color.title().replace('Gray', 'Grey') if color else None,
        'pattern': first_match(text, PATTERNS),
        'fabric': first_match(text, FABRICS),
        'fit': first_match(text, FITS),
    }

def aura_outfit(item_ids):
    # Reduce a Polyvore look to what AURA can build: dress or top+bottom, then 1 shoe, 1 layer, 1 accessory.
    pieces = [garment(i) for i in item_ids if i in INDEX]
    by_cat = {}
    for piece in pieces:
        by_cat.setdefault(piece['category'], []).append(piece)
    if 'Dress' in by_cat:
        core = [by_cat['Dress'][0]]
    elif 'Top' in by_cat and 'Bottom' in by_cat:
        core = [by_cat['Top'][0], by_cat['Bottom'][0]]
    else:
        return None
    extras = [by_cat[c][0] for c in ('Footwear', 'Outerwear', 'Accessory') if c in by_cat]
    return core + extras

OCCASIONS = [
    ('interview', ['interview', 'job interview']),
    ('wedding', ['wedding', 'bridesmaid', 'wedding guest']),
    ('meeting', ['office', 'work', 'business', 'meeting', 'workwear', '9 to 5', 'corporate']),
    ('date', ['date', 'date night', 'valentine', 'romantic']),
    ('party', ['party', 'night out', 'club', 'cocktail', 'new year', 'nye', 'prom', 'birthday']),
    ('travel', ['travel', 'airport', 'vacation', 'holiday', 'road trip', 'beach']),
    ('casual', ['casual', 'weekend', 'everyday', 'brunch', 'errands', 'school', 'college', 'campus']),
]
SEASONS = [('winter', 4, ['winter', 'snow', 'christmas', 'cozy']), ('autumn', 13, ['fall', 'autumn']),
           ('spring', 20, ['spring']), ('summer', 31, ['summer', 'beach', 'heat'])]

def title_of(set_id):
    value = outfit_titles.get(str(set_id), '')
    if isinstance(value, dict):
        value = f"{value.get('title', '')} {value.get('url_name', '')}"
    return str(value).lower().replace('-', ' ')

def occasion_of(title):
    for label, words in OCCASIONS:
        if any(re.search(rf'\b{re.escape(w)}\b', title) for w in words):
            return label
    return None

def season_of(title):
    for label, temperature, words in SEASONS:
        if any(re.search(rf'\b{re.escape(w)}\b', title) for w in words):
            condition = 'Snow' if 'snow' in title else 'Rain' if re.search(r'\brain', title) else 'Clear'
            return label, temperature, condition
    return None

def load_part(part):
    looks = []
    for outfit in json.load(open(DATA / SPLIT / f'{part}.json')):
        ordered = [e['item_id'] for e in sorted(outfit['items'], key=lambda e: e['index'])]
        look = aura_outfit(ordered)
        if look:
            title = title_of(outfit['set_id'])
            looks.append({'set_id': outfit['set_id'], 'items': look, 'occasion': occasion_of(title), 'season': season_of(title)})
    return looks

parts = {part: load_part(part) for part in ['train', 'valid', 'test']}
for part, looks in parts.items():
    print(part, len(looks), 'looks |', sum(l['occasion'] is not None for l in looks), 'with an occasion |',
          sum(l['season'] is not None for l in looks), 'with a season')


## 3. Build choose-the-best-option examples (real look vs. hard distractors)

In [ ]:
BODY_SHAPES = ['hourglass', 'rectangle', 'pear', 'apple', 'athletic']
SKIN_TONES = ['fair', 'light', 'medium', 'olive', 'tan', 'deep']
STYLES = ['minimalist', 'streetwear', 'formal', 'bohemian', 'athleisure', 'vintage', 'smart_casual', 'edgy']

def visual(looks):
    # Direct call (no model.predict overhead): thousands of small batches are scored here.
    packed = [[(EMB[INDEX[p['item_id']]], p['category']) for p in look] for look in looks]
    features, categories = encode_outfits(packed)
    logits = compat_model({'features': features, 'categories': categories}, training=False)
    logits = keras.ops.convert_to_numpy(logits).astype(np.float64).reshape(-1)
    return (1 / (1 + np.exp(-logits))).tolist()

def build_examples(looks, count, rng):
    pools = {}
    for look in looks:
        for piece in look['items']:
            pools.setdefault(piece['category'], []).append(piece)
    by_occasion, by_season = {}, {}
    for look in looks:
        if look['occasion']: by_occasion.setdefault(look['occasion'], []).append(look)
        if look['season']: by_season.setdefault(look['season'][0], []).append(look)
    with_context = [l for l in looks if l['occasion'] or l['season']]
    without = [l for l in looks if not (l['occasion'] or l['season'])]
    chosen = rng.sample(with_context, min(len(with_context), count // 2))
    chosen += rng.sample(without, min(len(without), count - len(chosen)))
    rng.shuffle(chosen)
    examples = []
    for look in chosen:
        target = look['items']
        def swap(n):
            # Standard Polyvore negatives: replace garments with random garments of the same category.
            outfit = [dict(p) for p in target]
            positions = rng.sample(range(len(outfit)), min(n, len(outfit)))
            for pos in positions:
                pool = [p for p in pools[outfit[pos]['category']] if p['item_id'] != outfit[pos]['item_id']]
                outfit[pos] = rng.choice(pool)
            return outfit
        distractors, contrast = [], None
        if look['occasion']:
            others = [o for o in by_occasion if o != look['occasion'] and by_occasion[o]]
            if others:
                other = rng.choice(others); distractors.append(rng.choice(by_occasion[other])['items']); contrast = ('occasion', other)
        if contrast is None and look['season']:
            opposite = {'winter': 'summer', 'summer': 'winter', 'autumn': 'summer', 'spring': 'winter'}[look['season'][0]]
            if by_season.get(opposite):
                distractors.append(rng.choice(by_season[opposite])['items']); contrast = ('season', opposite)
        distractors += [swap(1), swap(2)]
        if len(distractors) < 4:
            distractors.append(swap(len(target)))
        options = [target] + distractors[:4]
        order = list(range(len(options))); rng.shuffle(order)
        options = [options[i] for i in order]
        answer = order.index(0) + 1
        scores = visual(options)
        target_colors = {p['color'] for p in target if p['color']}
        other_colors = sorted({p['color'] for o in options for p in o if p['color']} - target_colors)
        preferences = {}
        if target_colors and rng.random() < 0.3:
            preferences['preferred_colors'] = rng.sample(sorted(target_colors), 1)
        if other_colors and rng.random() < 0.3:
            preferences['avoided_colors'] = rng.sample(other_colors, 1)
        if rng.random() < 0.2:
            preferences['preferred_styles'] = rng.sample(STYLES, 1)
        season = look['season']
        context = {
            'occasion': look['occasion'] or 'casual',
            'weather': {'temperature_c': season[1] if season else None, 'condition': season[2] if season else 'unknown'},
            'preferences': preferences,
        }
        if rng.random() < 0.6:
            context['body_shape'] = rng.choice(BODY_SHAPES)
            context['height_cm'] = int(rng.gauss(165, 9))
        if rng.random() < 0.4:
            context['skin_tone'] = rng.choice(SKIN_TONES)
        payload_options = [{'visual_match': s, 'items': o} for o, s in zip(options, scores)]
        reason = explain(target, answer, context, contrast, options, scores, look)
        examples.append({'messages': sp.build_messages(context, payload_options),
                         'answer': sp.format_answer(answer, reason), 'choice': answer,
                         'contrast': contrast[0] if contrast else 'visual', 'set_id': look['set_id'],
                         'avoided': preferences.get('avoided_colors', []), 'visual_scores': scores})
    return examples

def describe(piece):
    return ' '.join(v for v in [piece['color'], piece['type'].lower()] if v)

def explain(target, answer, context, contrast, options, scores, look):
    core = [describe(p) for p in target if p['category'] in ('Top', 'Bottom', 'Dress')]
    extra = [describe(p) for p in target if p['category'] not in ('Top', 'Bottom', 'Dress')]
    sentence = f"Option {answer} pairs the {' with the '.join(core)}"
    sentence += f", finished with the {' and '.join(extra[:2])}" if extra else ''
    colors = [p['color'] for p in target if p['color']]
    if len(set(colors)) >= 2:
        sentence += f"; the {', '.join(dict.fromkeys(colors))} palette holds together"
    if look['occasion']:
        sentence += f", and it reads right for {'a ' if look['occasion'] in ('date', 'party', 'wedding', 'meeting', 'interview') else ''}{look['occasion']}"
    elif look['season']:
        sentence += f", and it suits {look['season'][0]} weather"
    sentence += '.'
    if contrast and contrast[0] == 'occasion':
        sentence += f" Another option is a coherent look but is styled more for {contrast[1]}."
    elif contrast and contrast[0] == 'season':
        sentence += f" Another option is styled for {contrast[1]} rather than this weather."
    elif scores[answer - 1] >= max(scores) - 1e-9:
        sentence += ' Its pieces have the strongest visual match of the options.'
    avoided = context['preferences'].get('avoided_colors')
    if avoided:
        sentence += f" It also avoids {avoided[0].lower()}, which you prefer not to wear."
    return sentence

expected = {'train': N_TRAIN, 'valid': N_VALID, 'test': N_TEST}
saved = {name: OUT / f'{name}.jsonl' for name in expected}
if all(path.exists() for path in saved.values()):
    # Built in an earlier session: reuse exactly the same examples (the test set must not change).
    datasets = {name: [json.loads(line) for line in open(path)] for name, path in saved.items()}
    print('Reusing examples saved in Drive.')
else:
    rng = random.Random(SEED)
    datasets = {'train': build_examples(parts['train'], N_TRAIN, rng),
                'valid': build_examples(parts['valid'], N_VALID, rng),
                'test': build_examples(parts['test'], N_TEST, rng)}
    for name, rows in datasets.items():
        with open(saved[name], 'w') as handle:
            for row in rows: handle.write(json.dumps(row) + '\n')
    print(name, len(rows), 'examples | contrast types', {k: sum(r['contrast'] == k for r in rows) for k in ['occasion', 'season', 'visual']})
print(json.dumps(datasets['train'][0]['messages'][1]['content'])[:900])
print(datasets['train'][0]['answer'])


## 4. Baselines

In [ ]:
def accuracy(predictions, rows):
    by_type = {}
    for prediction, row in zip(predictions, rows):
        by_type.setdefault(row['contrast'], []).append(prediction == row['choice'])
    overall = float(np.mean([p == r['choice'] for p, r in zip(predictions, rows)]))
    return {'accuracy': overall, **{f'accuracy_{k}': float(np.mean(v)) for k, v in by_type.items()}}

test = datasets['test']
def visual_argmax(row):
    return int(np.argmax(row['visual_scores'])) + 1
EVAL_PATH = OUT / 'evaluation.json'
evaluation = json.loads(EVAL_PATH.read_text()) if EVAL_PATH.exists() else {}
evaluation['random'] = 1 / 5
evaluation['compatibility_model_only'] = accuracy([visual_argmax(r) for r in test], test)
def save_evaluation():
    EVAL_PATH.write_text(json.dumps(evaluation, indent=2))
save_evaluation()
print(json.dumps({k: v for k, v in evaluation.items() if k != 'training'}, indent=2))


## 5. Base Qwen2.5-1.5B-Instruct, zero-shot

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = 'left'
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='cuda')
print(torch.cuda.get_device_name(0), '| parameters', sum(p.numel() for p in model.parameters()) / 1e9, 'B')

@torch.no_grad()
def generate(model, rows, batch_size=8, max_new_tokens=140):
    model.eval(); outputs = []
    for start in range(0, len(rows), batch_size):
        prompts = [tokenizer.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=True)
                   for r in rows[start:start + batch_size]]
        enc = tokenizer(prompts, return_tensors='pt', padding=True, return_token_type_ids=False).to('cuda')
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        outputs += tokenizer.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return outputs

def score_generations(texts, rows):
    choices, valid, avoided_violations = [], 0, 0
    for text, row in zip(texts, rows):
        option_count = len(json.loads(row['messages'][1]['content'])['options'])
        try:
            choice, reason = sp.parse_answer(text, option_count); valid += 1
        except Exception:
            choice = -1
        choices.append(choice)
        if row['avoided'] and choice > 0:
            option = json.loads(row['messages'][1]['content'])['options'][choice - 1]
            avoided_violations += any(i.get('color') in row['avoided'] for i in option['items'])
    return {**accuracy(choices, rows), 'valid_json_rate': valid / len(rows),
            'avoided_colour_violations': avoided_violations}

EVAL_ROWS = test[:300]
if 'base_qwen_zero_shot' in evaluation:
    print('Base model already evaluated in an earlier session:', evaluation['base_qwen_zero_shot'])
else:
    began = time.time()
    base_texts = generate(model, EVAL_ROWS)
    evaluation['base_qwen_zero_shot'] = score_generations(base_texts, EVAL_ROWS)
    save_evaluation()
    print(f'{time.time() - began:.0f}s', json.dumps(evaluation['base_qwen_zero_shot'], indent=2))
    print('example:', base_texts[0][:300])


## 6. LoRA fine-tune (fp16 base, fp32 adapters, response-only loss)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments
from datasets import Dataset

MAX_LEN = 1024
def encode(row):
    prompt = tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=True)
    answer = row['answer'] + tokenizer.eos_token  # <|im_end|> closes the assistant turn
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(answer, add_special_tokens=False)['input_ids']
    ids = (prompt_ids + answer_ids)[-MAX_LEN:]
    labels = ([-100] * len(prompt_ids) + answer_ids)[-MAX_LEN:]
    return {'input_ids': ids, 'labels': labels, 'attention_mask': [1] * len(ids)}

train_ds = Dataset.from_list([encode(r) for r in datasets['train']])
valid_ds = Dataset.from_list([encode(r) for r in datasets['valid']])
print('max tokens', max(len(x) for x in train_ds['input_ids']))

def collate(batch):
    width = max(len(x['input_ids']) for x in batch)
    pad = tokenizer.pad_token_id or tokenizer.eos_token_id
    # Right padding for training (positions start at 0 for every sequence).
    return {
        'input_ids': torch.tensor([x['input_ids'] + [pad] * (width - len(x['input_ids'])) for x in batch]),
        'attention_mask': torch.tensor([x['attention_mask'] + [0] * (width - len(x['input_ids'])) for x in batch]),
        'labels': torch.tensor([x['labels'] + [-100] * (width - len(x['input_ids'])) for x in batch]),
    }

ADAPTER_DIR = OUT / 'lora_adapter'
TRAINING_DONE = (ADAPTER_DIR / 'adapter_config.json').exists()
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
if TRAINING_DONE:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR), is_trainable=False)
    print('Fine-tuned adapter found in Drive: training is skipped.')
else:
    model = get_peft_model(model, lora)
for parameter in (p for p in model.parameters() if p.requires_grad):
    parameter.data = parameter.data.float()   # fp32 adapters, fp16 frozen base (stable on T4)

args = TrainingArguments(
    output_dir=str(OUT / 'checkpoints'), num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine',
    warmup_ratio=0.03, per_device_train_batch_size=8, per_device_eval_batch_size=8, gradient_accumulation_steps=2,
    fp16=True, logging_steps=25, eval_strategy='steps', eval_steps=100, save_steps=100, save_total_limit=2,
    report_to='none', seed=SEED, remove_unused_columns=False, max_grad_norm=1.0,
)
if not TRAINING_DONE:
    model.print_trainable_parameters()
    # A checkpoint goes to Drive every 100 steps (~5-10 min); Run all resumes from the newest one.
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds, data_collator=collate)
    from transformers.trainer_utils import get_last_checkpoint
    last = get_last_checkpoint(str(OUT / 'checkpoints')) if (OUT / 'checkpoints').exists() else None
    print('Resuming from', last) if last else print('Starting training')
    began = time.time()
    train_result = trainer.train(resume_from_checkpoint=last)
    evaluation['training'] = {**train_result.metrics, 'final_eval_loss': trainer.evaluate()['eval_loss'],
                              'wall_minutes_this_session': (time.time() - began) / 60, 'examples': len(train_ds)}
    model.save_pretrained(str(ADAPTER_DIR))
    save_evaluation()
    print(json.dumps(evaluation['training'], indent=2))


## 7. Evaluate the fine-tuned stylist on held-out test looks

In [ ]:
model.config.use_cache = True
if 'fine_tuned' not in evaluation:
    tuned_texts = generate(model, EVAL_ROWS)
    evaluation['fine_tuned'] = score_generations(tuned_texts, EVAL_ROWS)
    save_evaluation()
    print('\nexample answer:', tuned_texts[0])
for name in ['compatibility_model_only', 'base_qwen_zero_shot', 'fine_tuned']:
    print(f"{name:26s}", {k: round(v, 3) for k, v in evaluation[name].items()})
print('random                    ', evaluation['random'])


## 8. Merge and export GGUF for the backend

In [ ]:
# Merge the adapter into the base model and export a 4-bit GGUF for llama.cpp in the backend.
import subprocess, shutil
GGUF = OUT / 'aura-stylist-q4_k_m.gguf'

def build_gguf(merged_dir):
    if not Path('/content/llama.cpp').exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', '/content/llama.cpp'], check=True)
    subprocess.run([sys.executable, '/content/llama.cpp/convert_hf_to_gguf.py', str(merged_dir),
                    '--outfile', '/content/aura-stylist-f16.gguf', '--outtype', 'f16'], check=True)
    subprocess.run('cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF '
                   '&& cmake --build /content/llama.cpp/build --target llama-quantize -j 4', shell=True, check=True)
    subprocess.run(['/content/llama.cpp/build/bin/llama-quantize', '/content/aura-stylist-f16.gguf',
                    '/content/aura-stylist-q4_k_m.gguf', 'Q4_K_M'], check=True)
    shutil.copy('/content/aura-stylist-q4_k_m.gguf', GGUF)

if GGUF.exists():
    print('GGUF already in Drive, nothing left to do:', GGUF)
else:
    merged_dir = Path('/content/aura-stylist-merged')
    merged = model.merge_and_unload()
    merged.save_pretrained(str(merged_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(merged_dir))
    del model, merged; torch.cuda.empty_cache()
    build_gguf(merged_dir)
    print('GGUF saved:', GGUF, round(GGUF.stat().st_size / 1e9, 2), 'GB')


## Install in the app
1. Download `MyDrive/AURA/stylist_llm/aura-stylist-q4_k_m.gguf` to `Capstone_Agents/model_artifacts/llm/aura-stylist-q4_k_m.gguf` (or set `AURA_STYLIST_LLM_PATH`).
2. Make sure `requirements-local.txt` is installed (llama-cpp-python), then restart the backend.
3. `GET /models/stylist/status` should report `fine_tuned_stylist`. Each `/recommend` response's `agent_trace` contains a `stylist_llm_agent` step showing which option the model chose, and the first outfit's `explanation_source` is `stylist_llm` when its reason passed the grounding checks.

## Reporting honestly
- `evaluation.json` compares random choice (20%), the compatibility model alone, base Qwen zero-shot and the fine-tuned model on 300 held-out Polyvore test examples, split by distractor type (`occasion`, `season`, `visual`).
- The occasion/season labels come from outfit titles, so they are noisy; the reasons are generated from templates over recorded attributes, so judge wording quality with a small human rating, not this accuracy alone.
- Body shape, height and skin tone are inputs the model reads, but there is no body-type ground truth in Polyvore — do not claim learned body-type expertise.
